In [ ]:
# resnet model

In [14]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split


In [18]:
# ============================
# Configuration
# ============================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset path — ကိုယ့် local dataset path ပြောင်းပါ
DATA_DIR = Path(r"D:\AI_Project_2026\AI_2026\Day12 CNN tf learning\groupA\po\food_class")

# Print dataset paths
print(f"DATA_DIR: {DATA_DIR}")


DATA_DIR: D:\AI_Project_2026\AI_2026\Day12 CNN tf learning\groupA\po\food_class


In [5]:
# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS_CLASSIFIER = 10
NUM_EPOCHS_FINETUNE = 5
LR_CLASSIFIER = 1e-3
LR_FINETUNE = 1e-4
VAL_RATIO = 0.15

# ResNet ImageNet normalization
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

NUM_WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 2)
PIN_MEMORY = DEVICE.type == "cuda"

# ResNet variant ရွေးပါ: "resnet18", "resnet34", "resnet50", "resnet101", "resnet152"
MODEL_NAME = "resnet50"

# Save paths
SAVE_DIR = Path("saved_models")

In [6]:

# ============================
# Dataset
# ============================
class ImageFolderDataset(Dataset):
    """
    Local folder structure: DATA_DIR/class_name/image.jpg
    """
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.class_names = sorted(
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        )
        self.class_to_idx = {c: i for i, c in enumerate(self.class_names)}

        for class_name in self.class_names:
            class_dir = os.path.join(root_dir, class_name)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    self.samples.append((
                        os.path.join(class_dir, fname),
                        self.class_to_idx[class_name]
                    ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


# ============================
# Transforms
# ============================
def get_train_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


def get_test_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])



In [23]:
# ============================
# DataLoaders
# ============================
def create_dataloaders(data_dir):
    # Load all data without transform for splitting
    all_dataset = ImageFolderDataset(data_dir, transform=None)
    
    # Split samples per class: 70% train, 15% val, 15% test
    train_samples = []
    val_samples = []
    test_samples = []
    
    for class_name in all_dataset.class_names:
        class_samples = [s for s in all_dataset.samples if s[1] == all_dataset.class_to_idx[class_name]]
        
        # Split: 70% train, 30% temp (then split temp into 15% val, 15% test)
        train_cls, temp = train_test_split(class_samples, test_size=0.3, random_state=42)
        val_cls, test_cls = train_test_split(temp, test_size=0.5, random_state=42)
        
        train_samples.extend(train_cls)
        val_samples.extend(val_cls)
        test_samples.extend(test_cls)
    
    # Create datasets with appropriate transforms
    train_dataset = ImageFolderDataset(data_dir, transform=get_train_transform())
    train_dataset.samples = train_samples
    train_dataset.class_names = all_dataset.class_names
    train_dataset.class_to_idx = all_dataset.class_to_idx
    
    val_dataset = ImageFolderDataset(data_dir, transform=get_test_transform())
    val_dataset.samples = val_samples
    val_dataset.class_names = all_dataset.class_names
    val_dataset.class_to_idx = all_dataset.class_to_idx
    
    test_dataset = ImageFolderDataset(data_dir, transform=get_test_transform())
    test_dataset.samples = test_samples
    test_dataset.class_names = all_dataset.class_names
    test_dataset.class_to_idx = all_dataset.class_to_idx
    
    # Create loaders
    loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)
    
    num_classes = len(all_dataset.class_names)
    class_names = all_dataset.class_names
    
    print(f"Classes: {num_classes} | Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")
    
    # Print class distribution
    from collections import Counter
    def print_class_dist(samples, class_names, split_name):
        labels = [s[1] for s in samples]
        counts = Counter(labels)
        print(f"\n{split_name} class distribution:")
        for i, class_name in enumerate(class_names):
            print(f"  {class_name}: {counts.get(i, 0)}")
    
    print_class_dist(train_samples, class_names, "Train")
    print_class_dist(val_samples, class_names, "Validation") 
    print_class_dist(test_samples, class_names, "Test")
    
    return train_loader, val_loader, test_loader, num_classes, class_names

In [24]:
# Create dataloaders
train_loader, val_loader, test_loader, num_classes, class_names = create_dataloaders(DATA_DIR)

Classes: 5 | Train: 3500 | Val: 750 | Test: 750

Train class distribution:
  chicken_wings: 700
  club_sandwich: 700
  donuts: 700
  pizza: 700
  sushi: 700

Validation class distribution:
  chicken_wings: 150
  club_sandwich: 150
  donuts: 150
  pizza: 150
  sushi: 150

Test class distribution:
  chicken_wings: 150
  club_sandwich: 150
  donuts: 150
  pizza: 150
  sushi: 150


In [26]:
# ============================
# Model — ResNet Family
# ============================
RESNET_VARIANTS = {
    "resnet18":  (models.resnet18,  models.ResNet18_Weights.IMAGENET1K_V1),
    "resnet34":  (models.resnet34,  models.ResNet34_Weights.IMAGENET1K_V1),
    "resnet50":  (models.resnet50,  models.ResNet50_Weights.IMAGENET1K_V2),
    "resnet101": (models.resnet101, models.ResNet101_Weights.IMAGENET1K_V2),
    "resnet152": (models.resnet152, models.ResNet152_Weights.IMAGENET1K_V2),
}


def create_model(num_classes, variant=MODEL_NAME):
    model_fn, weights = RESNET_VARIANTS[variant]
    model = model_fn(weights=weights)

    # Backbone freeze
    for param in model.parameters():
        param.requires_grad = False

    # Classifier head ပြောင်း
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, num_classes),
    )

    model = model.to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{variant}] Total: {total:,} | Trainable: {trainable:,} | Frozen: {total - trainable:,}")
    return model


In [27]:

# ============================
# Training Functions
# ============================
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


def train_loop(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, phase_name):
    best_val_acc = 0.0
    best_state = None

    print(f"\n{'='*60}")
    print(f"{phase_name}")
    print(f"{'='*60}")

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        marker = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            marker = " ✓"

        elapsed = time.time() - start
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
              f"{elapsed:.1f}s{marker}")

    if best_state:
        model.load_state_dict(best_state)
    print(f"Best Val Acc: {best_val_acc:.4f}")
    return best_val_acc


# ============================
# Phase 1: Classifier Only
# ============================
def train_classifier_only(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.fc.parameters(), lr=LR_CLASSIFIER)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)

    return train_loop(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        NUM_EPOCHS_CLASSIFIER, "Phase 1: Classifier Only (backbone frozen)"
    )


In [28]:

# ============================
# Phase 2: Full Fine-tune
# ============================
def finetune_full(model, train_loader, val_loader):
    # Unfreeze all layers
    for param in model.parameters():
        param.requires_grad = True

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam([
        {"params": [p for n, p in model.named_parameters() if "fc" not in n], "lr": LR_FINETUNE / 10},
        {"params": model.fc.parameters(), "lr": LR_FINETUNE},
    ])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Unfrozen all — trainable params: {trainable:,}")

    return train_loop(
        model, train_loader, val_loader, criterion, optimizer, scheduler,
        NUM_EPOCHS_FINETUNE, "Phase 2: Full Fine-tune (all layers)"
    )



In [29]:

# ============================
# Save Model
# ============================
def save_model(model, variant=MODEL_NAME):
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    # 1) Weights only
    weights_path = SAVE_DIR / f"weights_{variant}.pth"
    torch.save(model.state_dict(), weights_path)
    print(f"Weights saved: {weights_path} ({weights_path.stat().st_size / 1024 / 1024:.1f} MB)")

    # 2) Weights + model structure (full model)
    full_path = SAVE_DIR / f"full_{variant}.pth"
    torch.save(model, full_path)
    print(f"Full model saved: {full_path} ({full_path.stat().st_size / 1024 / 1024:.1f} MB)")

In [31]:


# ============================
# Main
# ============================
def main():
    print(f"Device: {DEVICE}")

    # 1. Data
    train_loader, val_loader, test_loader, num_classes, class_names = \
        create_dataloaders(DATA_DIR)

    # 2. Model
    model = create_model(num_classes, variant=MODEL_NAME)

    # 3. Classifier only training
    train_classifier_only(model, train_loader, val_loader)

    # 4. Full fine-tuning
    finetune_full(model, train_loader, val_loader)

    # 5. Test evaluation
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print(f"\nTest Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} ({test_acc*100:.1f}%)")

    # 6. Save
    save_model(model, variant=MODEL_NAME)


if __name__ == "__main__":
    main()


Device: cpu
Classes: 5 | Train: 3500 | Val: 750 | Test: 750

Train class distribution:
  chicken_wings: 700
  club_sandwich: 700
  donuts: 700
  pizza: 700
  sushi: 700

Validation class distribution:
  chicken_wings: 150
  club_sandwich: 150
  donuts: 150
  pizza: 150
  sushi: 150

Test class distribution:
  chicken_wings: 150
  club_sandwich: 150
  donuts: 150
  pizza: 150
  sushi: 150
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\saisa/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100.0%


[resnet50] Total: 24,559,685 | Trainable: 1,051,653 | Frozen: 23,508,032

Phase 1: Classifier Only (backbone frozen)
Epoch [ 1/10] train_loss=0.5578 train_acc=0.8117 | val_loss=0.2721 val_acc=0.9227 | 335.8s ✓
Epoch [ 2/10] train_loss=0.3212 train_acc=0.8914 | val_loss=0.2237 val_acc=0.9360 | 234.4s ✓
Epoch [ 3/10] train_loss=0.2579 train_acc=0.9109 | val_loss=0.2011 val_acc=0.9440 | 372.3s ✓
Epoch [ 4/10] train_loss=0.2099 train_acc=0.9306 | val_loss=0.2001 val_acc=0.9400 | 453.2s
Epoch [ 5/10] train_loss=0.1883 train_acc=0.9357 | val_loss=0.1983 val_acc=0.9427 | 391.5s
Epoch [ 6/10] train_loss=0.2085 train_acc=0.9309 | val_loss=0.1894 val_acc=0.9480 | 216.9s ✓
Epoch [ 7/10] train_loss=0.1842 train_acc=0.9337 | val_loss=0.2064 val_acc=0.9373 | 210.0s
Epoch [ 8/10] train_loss=0.1659 train_acc=0.9391 | val_loss=0.1822 val_acc=0.9480 | 205.2s
Epoch [ 9/10] train_loss=0.1774 train_acc=0.9377 | val_loss=0.2034 val_acc=0.9387 | 206.4s
Epoch [10/10] train_loss=0.1791 train_acc=0.9426 | val_l